In [ ]:
import os
import sys
from types import SimpleNamespace
from contextlib import nullcontext
import torch
from torch.utils.data import Dataset, DataLoader

# File: tests/minimal_train_test.py
# Minimal harness to run train_one_epoch from the training script with synthetic data.
# Adjust TRAIN_SCRIPT_CANDIDATES if your training script filename is different.

import importlib.util

import torch.nn as nn

# --- locate and import the training module (assumes one of these filenames) ---
TRAIN_SCRIPT_CANDIDATES = ['train.py', 'train_script.py', 'main.py']
train_path = None
for p in TRAIN_SCRIPT_CANDIDATES:
    if os.path.exists(p):
        train_path = os.path.abspath(p)
        break
if train_path is None:
    raise FileNotFoundError(f"Cannot find training script. Checked: {TRAIN_SCRIPT_CANDIDATES}")

spec = importlib.util.spec_from_file_location("train_module", train_path)
train_module = importlib.util.module_from_spec(spec)
sys.modules["train_module"] = train_module
spec.loader.exec_module(train_module)

# --- Monkeypatch a few heavy pieces in the training module to make a fast unit test ---
# Make batch_to_cuda a no-op for CPU testing
train_module.batch_to_cuda = lambda batch, device: batch

# Replace the heavy loss with a deterministic small tensor for backward
def fake_calculate_generator_token_loss(*args, **kwargs):
    return {
        "total_loss_for_backward": torch.tensor(0.1, requires_grad=True),
        "gen_loss": torch.tensor(0.05)
    }
train_module.calculate_generator_token_loss = fake_calculate_generator_token_loss

# Silence wandb in tests by forcing worker_args.wandb = False (we'll set worker_args below)

# Monkeypatch torch.amp.autocast to a no-op context manager so code that requests 'cuda' works on CPU.
# Save original to restore if needed.
_orig_autocast = torch.amp.autocast
torch.amp.autocast = lambda *a, **k: nullcontext()

# --- Dummy components mimicking the real classes enough for train_one_epoch to run ---

class DummyMaskDecoder:
    def __init__(self):
        # provide a token object with a weight attribute for calls in the training code
        self.hf_token_ar = SimpleNamespace(weight=torch.randn(1, 16))
        self.hf_token_tc = SimpleNamespace(weight=torch.randn(1, 16))
    def hf_mlp_ar(self, x):
        # accept tensor -> return a same-shaped tensor (pretend refined token)
        return x * 0.0
    def hf_mlp_tc(self, x):
        return x * 0.0

class DummyClimateSAM:
    def __init__(self, embed_dim=16):
        self.embed_dim = embed_dim
        self.mask_decoder = DummyMaskDecoder()
    def parameters(self):
        return []
    def encode_images(self, images):
        # images: tensor (B, C, H, W)
        B = images.shape[0]
        # Return fake embeddings, a list of intermediate features, and image_input & ori_img_size
        image_embeddings = torch.randn(B, self.embed_dim)  # dummy embeddings
        interm_features = [torch.randn(B, 8, images.shape[2]//4, images.shape[3]//4)]
        image_input = images  # pass-through
        ori_img_size = [(images.shape[2], images.shape[3])]
        return image_embeddings, interm_features, image_input, ori_img_size
    def forward(self, **kwargs):
        # return tc_pred_masks, ar_pred_masks, _  -> lists of per-image masks
        # produce empty lists sized to batch
        image_input = kwargs.get('image_input')
        B = image_input.shape[0]
        tc_masks = [torch.zeros(1, 1, image_input.shape[2], image_input.shape[3]) for _ in range(B)]
        ar_masks = [torch.zeros(1, 1, image_input.shape[2], image_input.shape[3]) for _ in range(B)]
        return tc_masks, ar_masks, None

class DummyPrompter(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        # simple parameter so optimizer has something to update
        self.fake_param = nn.Parameter(torch.randn(1))
        self.num_classes = num_classes
    def forward(self, interm_features, ar_refined=None, tc_refined=None):
        # interm_features is a list; infer batch and spatial size from first feature
        x = interm_features[0]
        B = x.shape[0]
        H = x.shape[2] * 4 if x.shape[2] > 0 else 32
        W = x.shape[3] * 4 if x.shape[3] > 0 else 32
        final_logit = torch.randn(B, self.num_classes, H, W)  # logits for multiclass (Background, TC, AR)
        interm_masks = None  # keep None to simplify
        ar_mask = None
        tc_mask = None
        return final_logit, interm_masks, ar_mask, tc_mask

class DummyPromptMaker:
    def make_prompts(self, multiclass_mask=None, ar_mask=None, tc_mask=None, enlarge_ratio=1.0):
        # Expect multiclass_mask shape (B, H, W) or tensor; produce lists for each prompt type
        if isinstance(multiclass_mask, torch.Tensor):
            B = multiclass_mask.shape[0]
        elif isinstance(multiclass_mask, list):
            B = len(multiclass_mask)
        else:
            B = 1
        # Each entry can be None or simple placeholder. The training code only indexes these lists.
        return {
            'ar_point_prompts': [None] * B,
            'tc_point_prompts': [None] * B,
            'ar_bbox_prompts': [None] * B,
            'tc_bbox_prompts': [None] * B,
            'ar_mask_prompts': [None] * B,
            'tc_mask_prompts': [None] * B,
        }

# --- Synthetic dataset matching expected batch dict structure ---
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=4, H=64, W=64):
        self.num_samples = num_samples
        self.H = H
        self.W = W
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        sample = {
            'input': torch.randn(3, self.H, self.W),
            # keep gt_mask as a single tensor per sample (later collate will make a list)
            'gt_mask': torch.randint(0, 3, (self.H, self.W), dtype=torch.long),
            'ar_centroids': None,
            'tc_centroids': None,
            'index_name': f"img_{idx}"
        }
        return sample

def collate_keep_masks(batch):
    # batch: list of sample dicts
    inputs = torch.stack([b['input'] for b in batch], dim=0)  # B, C, H, W
    gt_mask_list = [b['gt_mask'] for b in batch]  # keep as list of tensors (as training code expects)
    ar_centroids = [b['ar_centroids'] for b in batch]
    tc_centroids = [b['tc_centroids'] for b in batch]
    index_name = [b['index_name'] for b in batch]
    return {
        'input': inputs,
        'gt_mask': gt_mask_list,
        'ar_centroids': ar_centroids,
        'tc_centroids': tc_centroids,
        'index_name': index_name
    }

# --- Dummy scaler compatible with the training code usage ---
class DummyScaler:
    def scale(self, loss):
        # return the tensor itself so .backward() works
        return loss
    def step(self, optimizer):
        optimizer.step()
    def update(self):
        pass

# --- Build dataloader, models, optimizer, scheduler, and worker_args ---
dataset = SyntheticDataset(num_samples=4, H=32, W=32)
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_keep_masks)

device = 'cpu'
climatesam = DummyClimateSAM()
prompter = DummyPrompter()
prompt_maker = DummyPromptMaker()

# optimizer acts on prompter parameters (as in training code)
optimizer = torch.optim.SGD(prompter.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1.0)

scaler = DummyScaler()

worker_args = SimpleNamespace()
worker_args.prompt_enlarge_ratio = 1.0
worker_args.wandb = False
worker_args.exp_dir = "/tmp"
worker_args.run_name = "test_run"
worker_args.save_model = False

# --- Run one training epoch using the train_one_epoch function from the imported module ---
print("Starting minimal train_one_epoch test...")
train_module.train_one_epoch(
    epoch=1,
    train_dataloader=dataloader,
    climatesam=climatesam,
    prompter=prompter,
    prompt_maker=prompt_maker,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    local_rank=0,
    worker_args=worker_args,
    max_epoch_num=1,
    scaler=scaler,
    gradient_accumulation_steps=1
)
print("Completed minimal train_one_epoch test.")

# Restore original autocast (optional)
torch.amp.autocast = _orig_autocast